# Домашнее задание 6.1: Перенос импульса по кольцу

## Шаг 1. Расчётные формулы и численные схемы

Уравнение переноса и точное решение:
$$u_t + c u_x = 0, \quad x \in [0, 1), \quad c = 1, \quad u(x, t) = u_0\big((x - ct) \bmod 1\big).$$

Сетка: $x_j = j\Delta x$, $\Delta x = 1/N$, $j = 0, \dots, N-1$, $\Delta t = C\Delta x / c$.

1. **Upwind (1-й порядок):**
   $$u_j^{n+1} = u_j^n - C(u_j^n - u_{j-1}^n).$$
2. **Лакса–Вендроффа (2-й порядок):**
   $$u_j^{n+1} = u_j^n - \frac{C}{2}(u_{j+1}^n - u_{j-1}^n) + \frac{C^2}{2}(u_{j+1}^n - 2u_j^n + u_{j-1}^n).$$

Метрики (средняя ошибка $E_1$ и масса примеси $M$):
$$E_1 = \Delta x \sum_{j=0}^{N-1} |u_j^n - u(x_j, t_n)|, \quad M = \Delta x \sum_{j=0}^{N-1} u_j^n.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 11

# Схема против потока (upwind)
def step_upwind(u, C):
    N = len(u)
    u_next = np.zeros(N)
    for j in range(N):
        u_next[j] = u[j] - C * (u[j] - u[(j - 1) % N])
    return u_next

# Схема Лакса-Вендроффа
def step_lax_wendroff(u, C):
    N = len(u)
    u_next = np.zeros(N)
    for j in range(N):
        u_l = u[(j - 1) % N]
        u_r = u[(j + 1) % N]
        u_next[j] = u[j] - 0.5 * C * (u_r - u_l) + 0.5 * (C ** 2) * (u_r - 2.0 * u[j] + u_l)
    return u_next

# Точное решение на кольце [0, 1)
def get_exact(x, t, u0_func, c=1.0):
    xi = np.round((x - c * t) % 1.0, 12) % 1.0
    return u0_func(xi)

# Интегрирование по времени от t = 0 до t_end
def solve_advection(u0_func, N, C, t_end, scheme_func, c=1.0):
    dx = 1.0 / N
    x = np.arange(N) * dx
    dt = C * dx / c
    u = u0_func(x).copy()
    t = 0.0

    t_list, e1_list, m_list = [t], [0.0], [dx * np.sum(u)]
    min_list, max_list = [np.min(u)], [np.max(u)]

    while t < t_end - 1e-12:
        dt_step = min(dt, t_end - t)
        C_step = c * dt_step / dx
        u = scheme_func(u, C_step)
        t += dt_step

        u_ex = get_exact(x, t, u0_func, c)
        t_list.append(t)
        e1_list.append(dx * np.sum(np.abs(u - u_ex)))
        m_list.append(dx * np.sum(u))
        min_list.append(np.min(u))
        max_list.append(np.max(u))

    return {
        'x': x, 'dx': dx, 'u': u, 'u_ex': get_exact(x, t, u0_func, c),
        't': np.array(t_list), 'e1': np.array(e1_list), 'mass': np.array(m_list),
        'u_min': np.array(min_list), 'u_max': np.array(max_list)
    }

## Шаг 2. Часть 1: Гладкий профиль

Начальное условие: $u_0(x) = 1 + \frac{1}{2}\sin(2\pi x)$, $M_0 = 1$.
Параметры: $C = 0.8$, $N = 50, 100, 200$, $t = 1, 5$. Порядок сходимости: $p = \log_2(E_1(\Delta x) / E_1(\Delta x/2))$.

In [ ]:
def u0_smooth(x):
    return 1.0 + 0.5 * np.sin(2.0 * np.pi * x)

N_values = [50, 100, 200]
C_smooth = 0.8
results_smooth = {}

for t_end in [1.0, 5.0]:
    results_smooth[t_end] = {'upwind': {}, 'lw': {}}
    for N in N_values:
        results_smooth[t_end]['upwind'][N] = solve_advection(u0_smooth, N, C_smooth, t_end, step_upwind)
        results_smooth[t_end]['lw'][N] = solve_advection(u0_smooth, N, C_smooth, t_end, step_lax_wendroff)

for t_end in [1.0, 5.0]:
    print(f"=== Гладкий профиль: t = {t_end:.0f}, C = {C_smooth} ===")
    print("-" * 88)
    print(f"{'N':>5} | {'dx':>8} | {'E1 (Upwind)':>12} | {'Порядок p':>9} | {'M (Upwind)':>10} | {'E1 (Л-В)':>12} | {'Порядок p':>9} | {'M (Л-В)':>10}")
    print("-" * 88)
    prev_e1_up, prev_e1_lw = None, None
    for N in N_values:
        res_up = results_smooth[t_end]['upwind'][N]
        res_lw = results_smooth[t_end]['lw'][N]
        e1_up, e1_lw = res_up['e1'][-1], res_lw['e1'][-1]
        m_up, m_lw = res_up['mass'][-1], res_lw['mass'][-1]
        p_up_str = f"{np.log2(prev_e1_up / e1_up):9.2f}" if prev_e1_up else f"{'-':>9}"
        p_lw_str = f"{np.log2(prev_e1_lw / e1_lw):9.2f}" if prev_e1_lw else f"{'-':>9}"
        print(f"{N:5d} | {res_up['dx']:8.4f} | {e1_up:12.6f} | {p_up_str} | {m_up:10.6f} | {e1_lw:12.6f} | {p_lw_str} | {m_lw:10.6f}")
        prev_e1_up, prev_e1_lw = e1_up, e1_lw
    print("-" * 88 + "\n")

### Графики к части 1: профили ($t=1, 5$), ошибка $E_1(\Delta x)$ и масса $M(t)$

In [ ]:
# 1. Профили после 1 и 5 оборотов
x_fine = np.linspace(0.0, 1.0, 400, endpoint=False)
colors = {50: 'tab:red', 100: 'tab:orange', 200: 'tab:blue'}

for t_end in [1.0, 5.0]:
    plt.figure(figsize=(7, 5))
    plt.plot(x_fine, get_exact(x_fine, t_end, u0_smooth), 'k-', linewidth=2.0, label='Точное решение')
    for N in N_values:
        res_up = results_smooth[t_end]['upwind'][N]
        res_lw = results_smooth[t_end]['lw'][N]
        plt.plot(res_up['x'], res_up['u'], '--', color=colors[N], linewidth=1.5, label=f'Upwind (N={N})')
        plt.plot(res_lw['x'], res_lw['u'], ':', color=colors[N], linewidth=2.0, label=f'Лакс-Вендрофф (N={N})')
    plt.title(f'Профиль примеси при t = {int(t_end)} (C = 0.8)')
    plt.xlabel('x')
    plt.ylabel('u(x, t)')
    plt.xlim(0.0, 1.0)
    plt.ylim(0.4, 1.6)
    plt.grid(True)
    plt.legend(fontsize=9, ncol=2)
    plt.show()

# 2. Зависимость ошибки E1 от шага сетки dx
dx_vals = np.array([1.0 / N for N in N_values])
e1_up_t1 = np.array([results_smooth[1.0]['upwind'][N]['e1'][-1] for N in N_values])
e1_lw_t1 = np.array([results_smooth[1.0]['lw'][N]['e1'][-1] for N in N_values])
e1_up_t5 = np.array([results_smooth[5.0]['upwind'][N]['e1'][-1] for N in N_values])
e1_lw_t5 = np.array([results_smooth[5.0]['lw'][N]['e1'][-1] for N in N_values])

plt.figure(figsize=(7, 5))
plt.loglog(dx_vals, e1_up_t1, 'o-', color='tab:blue', label='Upwind (t = 1)')
plt.loglog(dx_vals, e1_up_t5, 's--', color='tab:blue', label='Upwind (t = 5)')
plt.loglog(dx_vals, e1_lw_t1, 'o-', color='tab:green', label='Лакс-Вендрофф (t = 1)')
plt.loglog(dx_vals, e1_lw_t5, 's--', color='tab:green', label='Лакс-Вендрофф (t = 5)')
plt.loglog(dx_vals, e1_up_t1[0] * (dx_vals / dx_vals[0]) ** 1, 'k:', alpha=0.6, label='O(dx)')
plt.loglog(dx_vals, e1_lw_t1[0] * (dx_vals / dx_vals[0]) ** 2, 'k-.', alpha=0.6, label='O(dx^2)')
plt.title('Ошибка E1(dx) на гладком профиле')
plt.xlabel('Шаг сетки dx')
plt.ylabel('E1')
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend(fontsize=9)
plt.show()

# 3. Сохранение массы примеси M(t)
res_up_50 = results_smooth[5.0]['upwind'][50]
res_lw_50 = results_smooth[5.0]['lw'][50]
res_up_200 = results_smooth[5.0]['upwind'][200]
res_lw_200 = results_smooth[5.0]['lw'][200]

plt.figure(figsize=(7, 5))
plt.plot(res_up_50['t'], res_up_50['mass'], '-', linewidth=2.0, label='Upwind (N=50)')
plt.plot(res_lw_50['t'], res_lw_50['mass'], '--', linewidth=2.0, label='Лакс-Вендрофф (N=50)')
plt.plot(res_up_200['t'], res_up_200['mass'], ':', linewidth=2.0, label='Upwind (N=200)')
plt.plot(res_lw_200['t'], res_lw_200['mass'], '-.', linewidth=1.5, label='Лакс-Вендрофф (N=200)')
plt.title('Количество примеси M(t) во времени')
plt.xlabel('t')
plt.ylabel('M(t)')
plt.ylim(0.95, 1.05)
plt.grid(True)
plt.legend()
plt.show()

## Шаг 3. Часть 2: Прямоугольный импульс и фронты

Начальное условие: $u_0(x) = 1$ при $0.2 \le x < 0.4$ и $0$ иначе ($M_0 = 0.2$).
Расчёт при $C = 0.5, 0.8, 1.0$ ($N = 200, t = 1$).

In [ ]:
def u0_rect(x):
    return np.where((x >= 0.2) & (x < 0.4), 1.0, 0.0)

N_rect = 200
C_stable_list = [0.5, 0.8, 1.0]
results_rect_C = {'upwind': {}, 'lw': {}}

for C_val in C_stable_list:
    results_rect_C['upwind'][C_val] = solve_advection(u0_rect, N_rect, C_val, 1.0, step_upwind)
    results_rect_C['lw'][C_val] = solve_advection(u0_rect, N_rect, C_val, 1.0, step_lax_wendroff)

print(f"=== Прямоугольный импульс: t = 1, N = {N_rect} ===")
print("-" * 88)
print(f"{'C':>5} | {'Схема':<16} | {'Ошибка E1':>10} | {'min(u)':>10} | {'max(u)':>10} | {'Масса M':>10}")
print("-" * 88)
for C_val in C_stable_list:
    for key, name in [('upwind', 'Upwind'), ('lw', 'Лакс-Вендрофф')]:
        r = results_rect_C[key][C_val]
        print(f"{C_val:5.1f} | {name:<16} | {r['e1'][-1]:10.5f} | {r['u_min'][-1]:10.4f} | {r['u_max'][-1]:10.4f} | {r['mass'][-1]:10.5f}")
    print("-" * 88)

### Полные профили ($C = 0.5, 0.8, 1.0$) и увеличенные фрагменты фронтов в одинаковом масштабе

In [ ]:
# Полные профили при C = 0.5, 0.8, 1.0
for C_val in C_stable_list:
    r_up = results_rect_C['upwind'][C_val]
    r_lw = results_rect_C['lw'][C_val]
    plt.figure(figsize=(7, 5))
    plt.plot(r_up['x'], r_up['u_ex'], 'k-', linewidth=2.0, label='Точное решение')
    plt.plot(r_up['x'], r_up['u'], '--', color='tab:blue', linewidth=1.8, label='Upwind')
    plt.plot(r_lw['x'], r_lw['u'], '-', color='tab:red', linewidth=1.5, label='Лакс-Вендрофф')
    plt.title(f'Прямоугольный импульс при t = 1 (N = {N_rect}, C = {C_val})')
    plt.xlabel('x')
    plt.ylabel('u(x, t)')
    plt.xlim(0.0, 1.0)
    plt.ylim(-0.3, 1.3)
    plt.grid(True)
    plt.legend()
    plt.show()

# Увеличенные фрагменты заднего (x = 0.2) и переднего (x = 0.4) фронтов (окно dx = 0.24, y в [-0.3, 1.3])
for C_val in [0.5, 0.8]:
    r_up = results_rect_C['upwind'][C_val]
    r_lw = results_rect_C['lw'][C_val]

    for x_center, front_name, xlims in [(0.2, 'Задний фронт (x = 0.2)', (0.08, 0.32)),
                                        (0.4, 'Передний фронт (x = 0.4)', (0.28, 0.52))]:
        plt.figure(figsize=(7, 5))
        plt.plot(r_up['x'], r_up['u_ex'], 'k-', linewidth=2.0, label='Точное решение')
        plt.plot(r_up['x'], r_up['u'], 'o--', color='tab:blue', markersize=4, label='Upwind')
        plt.plot(r_lw['x'], r_lw['u'], 's-', color='tab:red', markersize=4, label='Лакс-Вендрофф')
        plt.title(f'{front_name}, C = {C_val}, t = 1')
        plt.xlabel('x')
        plt.ylabel('u(x, t)')
        plt.xlim(*xlims)
        plt.ylim(-0.3, 1.3)
        plt.grid(True)
        plt.legend()
        plt.show()

## Шаг 4. Сгущение сетки на прямоугольном импульсе ($C = 0.8, t = 1$)

In [ ]:
N_grid_rect = [50, 100, 200, 400]
res_grid_up, res_grid_lw = {}, {}

for N in N_grid_rect:
    res_grid_up[N] = solve_advection(u0_rect, N, 0.8, 1.0, step_upwind)
    res_grid_lw[N] = solve_advection(u0_rect, N, 0.8, 1.0, step_lax_wendroff)

print("=== Сгущение сетки на прямоугольном импульсе (C = 0.8, t = 1) ===")
print("-" * 98)
print(f"{'N':>5} | {'dx':>8} | {'E1 (Upw)':>10} | {'p (Upw)':>8} | {'[min, max] Upwind':>18} | {'E1 (Л-В)':>10} | {'p (Л-В)':>8} | {'[min, max] Л-В':>18}")
print("-" * 98)
prev_up, prev_lw = None, None
for N in N_grid_rect:
    ru, rl = res_grid_up[N], res_grid_lw[N]
    e1_u, e1_l = ru['e1'][-1], rl['e1'][-1]
    pu = f"{np.log2(prev_up / e1_u):8.2f}" if prev_up else f"{'-':>8}"
    pl = f"{np.log2(prev_lw / e1_l):8.2f}" if prev_lw else f"{'-':>8}"
    print(f"{N:5d} | {ru['dx']:8.4f} | {e1_u:10.5f} | {pu} | [{ru['u_min'][-1]:6.3f}, {ru['u_max'][-1]:6.3f}] | {e1_l:10.5f} | {pl} | [{rl['u_min'][-1]:6.3f}, {rl['u_max'][-1]:6.3f}]")
    prev_up, prev_lw = e1_u, e1_l
print("-" * 98)

dx_rect = np.array([1.0 / N for N in N_grid_rect])
e1_rect_up = np.array([res_grid_up[N]['e1'][-1] for N in N_grid_rect])
e1_rect_lw = np.array([res_grid_lw[N]['e1'][-1] for N in N_grid_rect])

plt.figure(figsize=(7, 5))
plt.loglog(dx_rect, e1_rect_up, 'o-', color='tab:blue', label='Upwind (p ~ 0.50)')
plt.loglog(dx_rect, e1_rect_lw, 's-', color='tab:red', label='Лакс-Вендрофф (p ~ 0.58)')
plt.title('Ошибка E1(dx) на прямоугольном импульсе (C = 0.8)')
plt.xlabel('Шаг сетки dx')
plt.ylabel('E1')
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(N_grid_rect, [res_grid_up[N]['u_max'][-1] for N in N_grid_rect], 'o--', color='tab:blue', label='max(u) Upwind')
plt.plot(N_grid_rect, [res_grid_up[N]['u_min'][-1] for N in N_grid_rect], 's--', color='tab:cyan', label='min(u) Upwind')
plt.plot(N_grid_rect, [res_grid_lw[N]['u_max'][-1] for N in N_grid_rect], 'o-', color='tab:red', label='max(u) Лакс-Вендрофф')
plt.plot(N_grid_rect, [res_grid_lw[N]['u_min'][-1] for N in N_grid_rect], 's-', color='tab:orange', label='min(u) Лакс-Вендрофф')
plt.axhline(1.0, color='k', linestyle=':', alpha=0.6)
plt.axhline(0.0, color='k', linestyle=':', alpha=0.6)
plt.title('Экстремумы min(u) и max(u) от числа узлов N (t = 1, C = 0.8)')
plt.xlabel('N')
plt.ylabel('min(u), max(u)')
plt.grid(True)
plt.legend()
plt.show()

## Шаг 5. Динамика $\min(u)$, $\max(u)$, $E_1(t)$ и неустойчивость при $C = 1.1$

Условие устойчивости Куранта: $C \le 1$. При $C = 1.1$ гармоника $k\Delta x = \pi$ растёт на каждом шаге ($|G(\pi)| = |1 - 2C| = 1.2 > 1$ для upwind и $|G(\pi)| = |1 - 2C^2| = 1.42 > 1$ для Лакса–Вендроффа).

In [ ]:
# 1. Устойчивые режимы (C = 0.5, 0.8, 1.0)
for metric_key, ylabel_str, title_str in [
    ('u_max', 'max(u)', 'Максимум max(u) во времени (устойчивые C)'),
    ('u_min', 'min(u)', 'Минимум min(u) во времени (устойчивые C)'),
    ('e1', 'E1(t)', 'Ошибка E1(t) во времени (устойчивые C)')
]:
    plt.figure(figsize=(7, 5))
    for C_val, col in zip([0.5, 0.8, 1.0], ['tab:blue', 'tab:green', 'tab:purple']):
        r_up = results_rect_C['upwind'][C_val]
        r_lw = results_rect_C['lw'][C_val]
        plt.plot(r_up['t'], r_up[metric_key], '--', color=col, label=f'Upwind (C={C_val})')
        plt.plot(r_lw['t'], r_lw[metric_key], '-', color=col, label=f'Л-В (C={C_val})')
    plt.title(title_str)
    plt.xlabel('t')
    plt.ylabel(ylabel_str)
    plt.grid(True)
    plt.legend(fontsize=9, ncol=2)
    plt.show()

# 2. Неустойчивый режим C = 1.1 (N = 100)
N_unst, C_unst = 100, 1.1
res_unst_up = solve_advection(u0_rect, N_unst, C_unst, 0.55, step_upwind)
res_unst_lw = solve_advection(u0_rect, N_unst, C_unst, 0.55, step_lax_wendroff)

print(f"=== Развитие неустойчивости при C = {C_unst} (N = {N_unst}) ===")
print("-" * 86)
print(f"{'Шаг n':>6} | {'Время t':>8} | {'max(u) Upwind':>14} | {'E1 Upwind':>12} | {'max(u) Л-В':>14} | {'E1 Л-В':>12}")
print("-" * 86)
for step_idx in [0, 10, 20, 30, 40, 50]:
    t_val = res_unst_up['t'][step_idx]
    print(f"{step_idx:6d} | {t_val:8.3f} | {res_unst_up['u_max'][step_idx]:14.4e} | {res_unst_up['e1'][step_idx]:12.4e} | {res_unst_lw['u_max'][step_idx]:14.4e} | {res_unst_lw['e1'][step_idx]:12.4e}")
print("-" * 86)

res_unst_up_20 = solve_advection(u0_rect, N_unst, C_unst, 0.22, step_upwind)
res_unst_lw_20 = solve_advection(u0_rect, N_unst, C_unst, 0.22, step_lax_wendroff)

plt.figure(figsize=(7, 5))
plt.plot(res_unst_up_20['x'], res_unst_up_20['u_ex'], 'k-', linewidth=2.0, label='Точное решение')
plt.plot(res_unst_up_20['x'], res_unst_up_20['u'], '--', color='tab:blue', label='Upwind (C=1.1, 20 шагов)')
plt.plot(res_unst_lw_20['x'], res_unst_lw_20['u'], '-', color='tab:red', alpha=0.85, label='Лакс-Вендрофф (C=1.1, 20 шагов)')
plt.title('Разрушение решения при C = 1.1 (t = 0.22, N = 100)')
plt.xlabel('x')
plt.ylabel('u(x, t)')
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(7, 5))
plt.semilogy(res_unst_up['t'][1:], res_unst_up['u_max'][1:], 'o--', color='tab:blue', markersize=4, label='max(u) Upwind')
plt.semilogy(res_unst_up['t'][1:], np.abs(res_unst_up['u_min'][1:]), 's:', color='tab:cyan', markersize=4, label='|min(u)| Upwind')
plt.semilogy(res_unst_lw['t'][1:], res_unst_lw['u_max'][1:], 'o-', color='tab:red', markersize=4, label='max(u) Лакс-Вендрофф')
plt.semilogy(res_unst_lw['t'][1:], np.abs(res_unst_lw['u_min'][1:]), 's-.', color='tab:orange', markersize=4, label='|min(u)| Лакс-Вендрофф')
plt.title('Рост экстремумов при C = 1.1 (неустойчивый режим)')
plt.xlabel('t')
plt.ylabel('Амплитуда (лог. шкала)')
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend()
plt.show()

plt.figure(figsize=(7, 5))
plt.semilogy(res_unst_up['t'][1:], res_unst_up['e1'][1:], 'o--', color='tab:blue', markersize=4, label='E1(t) Upwind (C=1.1)')
plt.semilogy(res_unst_lw['t'][1:], res_unst_lw['e1'][1:], 's-', color='tab:red', markersize=4, label='E1(t) Лакс-Вендрофф (C=1.1)')
plt.title('Рост ошибки E1(t) при C = 1.1 (неустойчивый режим)')
plt.xlabel('t')
plt.ylabel('E1(t) (лог. шкала)')
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend()
plt.show()

## Шаг 6. Анимация переноса прямоугольного импульса ($N = 200, C = 0.8$)

In [ ]:
N_anim, C_anim = 200, 0.8
dx_anim = 1.0 / N_anim
x_anim = np.arange(N_anim) * dx_anim
dt_anim = C_anim * dx_anim

u_up_curr = u0_rect(x_anim).copy()
u_lw_curr = u0_rect(x_anim).copy()

frames_t, frames_ex, frames_up, frames_lw = [0.0], [u0_rect(x_anim).copy()], [u_up_curr.copy()], [u_lw_curr.copy()]

for step in range(1, int(round(1.0 / dt_anim)) + 1):
    u_up_curr = step_upwind(u_up_curr, C_anim)
    u_lw_curr = step_lax_wendroff(u_lw_curr, C_anim)
    if step % 5 == 0:
        t_now = step * dt_anim
        frames_t.append(t_now)
        frames_ex.append(get_exact(x_anim, t_now, u0_rect))
        frames_up.append(u_up_curr.copy())
        frames_lw.append(u_lw_curr.copy())

fig_anim, ax_anim = plt.subplots(figsize=(7, 5))
line_ex, = ax_anim.plot(x_anim, frames_ex[0], 'k-', linewidth=2.0, label='Точное решение')
line_up, = ax_anim.plot(x_anim, frames_up[0], '--', color='tab:blue', linewidth=1.8, label='Upwind')
line_lw, = ax_anim.plot(x_anim, frames_lw[0], '-', color='tab:red', linewidth=1.5, label='Лакс-Вендрофф')
title_anim = ax_anim.set_title('Перенос импульса: t = 0.00 (C = 0.8)')
ax_anim.set_xlabel('x')
ax_anim.set_ylabel('u(x, t)')
ax_anim.set_xlim(0.0, 1.0)
ax_anim.set_ylim(-0.3, 1.3)
ax_anim.grid(True)
ax_anim.legend(loc='upper right')

def update_frame(k):
    line_ex.set_ydata(frames_ex[k])
    line_up.set_ydata(frames_up[k])
    line_lw.set_ydata(frames_lw[k])
    title_anim.set_text(f'Перенос импульса: t = {frames_t[k]:.2f} (C = 0.8)')
    return line_ex, line_up, line_lw, title_anim

anim = FuncAnimation(fig_anim, update_frame, frames=len(frames_t), interval=80, blit=True)

# Вывод анимации прямо в ноутбук (без сохранения в файл)
plt.close(fig_anim)
HTML(anim.to_jshtml())

## Выводы и ответы на вопросы

1. **Гладкий профиль:** схема upwind сходится с 1-м порядком ($p \approx 1.0$), схема Лакса–Вендроффа — со 2-м порядком ($p \approx 2.0$).
2. **Означает ли сохранение количества примеси, что её распределение найдено правильно?**
   Нет. Обе схемы консервативны и сохраняют массу $M = 1$ с машинной точностью ($\sim 10^{-16}$), но схема upwind сильно гасит амплитуду волны из-за численной вязкости, перераспределяя примесь из максимумов в минимумы.
3. **Почему схема, более точная на гладком профиле, даёт нефизические значения на разрыве?**
   По теореме Годунова линейная схема порядка выше первого не может быть монотонной. В схеме upwind главная ошибка чётного порядка $\frac{c\Delta x}{2}(1-C)u_{xx}$ (численная диффузия) монотонно сглаживает фронт ($0 \le u \le 1$). В схеме Лакса–Вендроффа главная ошибка нечётного порядка $-\frac{c\Delta x^2}{6}(1-C^2)u_{xxx}$ (численная дисперсия) вызывает разбегание гармоник с разной фазовой скоростью и приводит к осцилляциям Гиббса ($\min u \approx -0.2$, $\max u \approx 1.2$), не исчезающим при сгущении сетки.
4. **Почему случай $C = 1$ требует отдельного рассмотрения?**
   При $C = 1$ обе схемы вырождаются в точный поузловой сдвиг $u_j^{n+1} = u_{j-1}^n$, а коэффициенты диффузии и дисперсии обращаются в нуль ($E_1 = 0$). Этот режим лежит на границе устойчивости ($C \le 1$) и недостижим во всей области в задачах с переменной скоростью или неравномерной сеткой, поэтому не отражает реальных свойств схем при $0 < C < 1$.